# Nexus — HyperShelf: All-Store Pipeline
## Objectives 3 & 4 + Phantom Detection — Runs Across ALL 478 Stores
> **What this notebook does:** Loops through every store in the dataset, computes optimized safety stock,
> reorder points, live alert tiers, phantom inventory, and exports per-store CSVs + a network-level master file.
> No manual store selection needed — fully automated end-to-end.

---
## Section 0 — Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for batch runs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from pathlib import Path
from scipy import stats
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── UPDATE THIS PATH ─────────────────────────────────────────────
DATA_DIR   = Path('/Users/shashi/retail-ai-project/data/raw/output/csv')
OUTPUT_DIR = Path('/Users/shashi/retail-ai-project/data/processed/nexus/allstore')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── SERVICE LEVEL ────────────────────────────────────────────────
SERVICE_LEVEL = 0.975
Z_SCORE       = stats.norm.ppf(SERVICE_LEVEL)  # 1.9600

# ── ALERT THRESHOLDS ─────────────────────────────────────────────
CRITICAL_DAYS = 3
WARNING_DAYS  = 7
MONITOR_DAYS  = 14
TARGET_DAYS   = 14

# ── PHANTOM DETECTION ────────────────────────────────────────────
PHANTOM_SLOW_MOVER_PCT = 0.05

# ── PRIORITY WEIGHTS ─────────────────────────────────────────────
TIER_WEIGHT   = {'CRITICAL': 1000, 'WARNING': 100, 'MONITOR': 10, 'OK': 0}
REV_RISK_DIV  = 1000
DOS_WEIGHT    = 5
SUPP_WEIGHT   = 20
SEASON_WEIGHT = 10

NOW    = datetime.now()
TODAY  = pd.Timestamp.today().normalize()
LOOKAHEAD = TODAY + timedelta(days=7)

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print('='*62)
print('  NEXUS | ALL-STORE PIPELINE')
print('='*62)
print(f'  Generated : {NOW.strftime("%A, %B %d, %Y at %I:%M %p")}')
print(f'  Z Score   : {Z_SCORE:.4f}  ({SERVICE_LEVEL*100:.1f}% service level)')
print(f'  Output Dir: {OUTPUT_DIR.resolve()}')
print('='*62)


  NEXUS | ALL-STORE PIPELINE
  Generated : Saturday, April 25, 2026 at 12:52 PM
  Z Score   : 1.9600  (97.5% service level)
  Output Dir: /Users/shashi/retail-ai-project/data/processed/nexus/allstore


---
## Section 1 — Load All Datasets (Once)
> Sales loaded in 500K-row chunks and pre-aggregated to daily store-SKU level before looping.
> Everything else is loaded once and filtered per store inside the loop.

In [2]:
# ── Reference tables ─────────────────────────────────────────────
stores     = pd.read_csv(DATA_DIR / 'stores.csv',    parse_dates=['open_date'])
products   = pd.read_csv(DATA_DIR / 'products.csv')
suppliers  = pd.read_csv(DATA_DIR / 'suppliers.csv', parse_dates=['contract_start'])
inv_snap   = pd.read_csv(DATA_DIR / 'inventory_snapshots.csv',
                          parse_dates=['snapshot_date','expiry_nearest_date'])
repl_logs  = pd.read_csv(DATA_DIR / 'replenishment_logs.csv',
                          parse_dates=['replenishment_date','order_date','receive_date'])
stockout   = pd.read_csv(DATA_DIR / 'stockout_events.csv',
                          parse_dates=['stockout_date','restock_date'])
promotions = pd.read_csv(DATA_DIR / 'promotions.csv',
                          parse_dates=['start_date','end_date'])

# ── Sales — chunked load & pre-aggregated ─────────────────────────
print('Loading sales_transactions.csv in chunks...')
chunks = []
for chunk in pd.read_csv(DATA_DIR / 'sales_transactions.csv',
                          parse_dates=['sale_date'], chunksize=500_000):
    grp = (chunk.groupby(['store_id','sku_id','sale_date'])
               .agg(units_sold=('units_sold','sum'), revenue=('revenue','sum'))
               .reset_index())
    chunks.append(grp)

sales = (pd.concat(chunks)
           .groupby(['store_id','sku_id','sale_date'])
           .agg(units_sold=('units_sold','sum'), revenue=('revenue','sum'))
           .reset_index())

sales['month']  = sales['sale_date'].dt.month
sales['dow']    = sales['sale_date'].dt.dayofweek
sales['week']   = sales['sale_date'].dt.to_period('W')

# Pre-aggregate weekly sales for phantom detection
sales_weekly = (sales.groupby(['store_id','sku_id','week'])
                    .agg(weekly_units=('units_sold','sum'))
                    .reset_index())

# Product-supplier enrichment table (built once)
prod_sup = products.merge(
    suppliers[['supplier_id','supplier_name','lead_time_days_avg',
               'lead_time_days_std','reliability_score']],
    on='supplier_id', how='left').drop_duplicates(subset='sku_id', keep='first')  # FIX: prevent duplicate merge crash

# Active promotions window
active_promos = promotions[
    (promotions['start_date'] <= LOOKAHEAD) &
    (promotions['end_date']   >= TODAY)
].copy()

ALL_STORE_IDS = sorted(stores['store_id'].unique())
print(f'Loaded. {len(ALL_STORE_IDS)} stores to process.')
print(f'Sales rows: {len(sales):,} | Snapshots: {len(inv_snap):,} | Replenishments: {len(repl_logs):,}')


Loading sales_transactions.csv in chunks...
Loaded. 478 stores to process.
Sales rows: 25,969,106 | Snapshots: 4,015,200 | Replenishments: 1,366,761


---
## Section 2 — Per-Store Processing Functions
> Each function below handles one step. The main loop calls them in sequence for every store.
> Functions are pure — they only take store-filtered dataframes and return results.

In [3]:
def compute_seasonal(s_sales, today=TODAY):
    """Returns monthly_idx dict and seasonal_factor for current month."""
    if len(s_sales) == 0:
        return {}, 1.0
    monthly_avg = s_sales.groupby('month')['units_sold'].mean()
    overall_avg = s_sales['units_sold'].mean()
    if overall_avg == 0:
        return {}, 1.0
    monthly_idx = (monthly_avg / overall_avg * 100).round(1).to_dict()
    seasonal_factor = monthly_idx.get(today.month, 100) / 100
    return monthly_idx, seasonal_factor


def compute_demand_stats(s_sales, prod_sup, seasonal_factor):
    """Demand stats per SKU enriched with product/supplier info."""
    if len(s_sales) == 0:
        return pd.DataFrame()
    demand_stats = (
        s_sales.groupby('sku_id')
        .agg(
            avg_daily_demand = ('units_sold', 'mean'),
            std_daily_demand = ('units_sold', 'std'),
            max_daily_demand = ('units_sold', 'max'),
            total_units      = ('units_sold', 'sum'),
            total_revenue    = ('revenue',    'sum'),
        )
        .reset_index()
    )
    demand_stats['std_daily_demand'] = demand_stats['std_daily_demand'].fillna(0)
    demand_stats = demand_stats.merge(
        prod_sup[['sku_id','product_name','category','subcategory',
                  'unit_price','unit_cost','pack_size','reorder_point',
                  'safety_stock','is_perishable','shelf_life_days',
                  'supplier_id','supplier_name','lead_time_days_avg',
                  'lead_time_days_std','reliability_score']],
        on='sku_id', how='left')
    demand_stats['seasonal_factor']      = seasonal_factor
    demand_stats['avg_demand_seasonal']  = demand_stats['avg_daily_demand'] * seasonal_factor
    return demand_stats


def compute_lead_times(s_repl, demand_stats):
    """Merge actual lead times from replenishment logs into demand_stats."""
    if len(s_repl) == 0:
        demand_stats['lead_time_final']     = demand_stats['lead_time_days_avg'].fillna(7)
        demand_stats['lead_time_std_final'] = demand_stats['lead_time_days_std'].fillna(1)
        demand_stats['avg_fill_rate']       = 1.0
        return demand_stats

    s_repl = s_repl.copy()
    s_repl['fulfillment_rate'] = s_repl['units_received'] / s_repl['units_ordered'].replace(0, np.nan)
    lead_actual = (
        s_repl.groupby('sku_id')
        .agg(
            avg_lead_actual = ('lead_time_actual', 'mean'),
            std_lead_actual = ('lead_time_actual', 'std'),
            avg_fill_rate   = ('fulfillment_rate',  'mean'),
        )
        .reset_index()
    )
    lead_actual['std_lead_actual'] = lead_actual['std_lead_actual'].fillna(0)
    demand_stats = demand_stats.merge(
        lead_actual[['sku_id','avg_lead_actual','std_lead_actual','avg_fill_rate']],
        on='sku_id', how='left')
    demand_stats['lead_time_final']     = demand_stats['avg_lead_actual'].fillna(demand_stats['lead_time_days_avg'])
    demand_stats['lead_time_std_final'] = demand_stats['std_lead_actual'].fillna(demand_stats['lead_time_days_std'].fillna(0))
    demand_stats['avg_fill_rate']       = demand_stats['avg_fill_rate'].fillna(1.0)
    g_lt  = demand_stats['lead_time_final'].mean()
    g_std = demand_stats['lead_time_std_final'].mean()
    demand_stats['lead_time_final']     = demand_stats['lead_time_final'].fillna(g_lt if not np.isnan(g_lt) else 7)
    demand_stats['lead_time_std_final'] = demand_stats['lead_time_std_final'].fillna(g_std if not np.isnan(g_std) else 1)
    return demand_stats


def compute_safety_stock(demand_stats, Z_SCORE, seasonal_factor, TODAY=TODAY):
    """Objective 3: safety stock + ROP formula."""
    demand_stats['safety_stock_optimized'] = np.ceil(
        Z_SCORE * np.sqrt(
            demand_stats['lead_time_final']     * demand_stats['std_daily_demand']**2 +
            demand_stats['avg_daily_demand']**2 * demand_stats['lead_time_std_final']**2
        )
    ).clip(lower=0).astype(int)
    demand_stats['safety_stock_seasonal']    = np.ceil(
        demand_stats['safety_stock_optimized'] * seasonal_factor).astype(int)
    demand_stats['reorder_point_optimized']  = np.ceil(
        demand_stats['avg_daily_demand'] * demand_stats['lead_time_final'] +
        demand_stats['safety_stock_seasonal']).astype(int)
    demand_stats['ss_delta']  = demand_stats['safety_stock_seasonal'] - demand_stats['safety_stock']
    demand_stats['rop_delta'] = demand_stats['reorder_point_optimized'] - demand_stats['reorder_point']
    demand_stats['ss_status'] = demand_stats['ss_delta'].apply(
        lambda d: 'UNDERSTOCKED' if d > 10 else ('OVERSTOCKED' if d < -10 else 'OPTIMAL'))
    return demand_stats


def compute_phantom(s_snap, s_weekly, products, PHANTOM_SLOW_MOVER_PCT=0.05):
    """Phantom inventory detection for one store."""
    if len(s_snap) == 0 or len(s_weekly) == 0:
        return pd.DataFrame(), set(), 0.0
    s_snap = s_snap.copy()
    s_snap['week'] = s_snap['snapshot_date'].dt.to_period('W')
    snap_weekly = (
        s_snap.groupby(['sku_id','week'])
        .agg(avg_on_hand=('units_on_hand','mean'), avg_backroom=('units_in_backroom','mean'))
        .reset_index()
    )
    phantom_df = snap_weekly.merge(
        s_weekly[['sku_id','week','weekly_units']], on=['sku_id','week'], how='left')
    phantom_df['weekly_units'] = phantom_df['weekly_units'].fillna(0)
    phantom_df['is_phantom_candidate'] = (
        (phantom_df['avg_on_hand'] > 0) & (phantom_df['weekly_units'] == 0))
    sku_avg = s_weekly.groupby('sku_id')['weekly_units'].mean().reset_index().rename(
        columns={'weekly_units':'sku_avg_weekly'})
    phantom_df = phantom_df.merge(sku_avg, on='sku_id', how='left')
    threshold = phantom_df['sku_avg_weekly'].quantile(PHANTOM_SLOW_MOVER_PCT)
    phantom_df.loc[phantom_df['sku_avg_weekly'] < threshold, 'is_phantom_candidate'] = False
    phantom_events = phantom_df[phantom_df['is_phantom_candidate']].copy()
    if len(phantom_events) == 0:
        return pd.DataFrame(), set(), 0.0
    phantom_events = phantom_events.merge(
        products[['sku_id','product_name','category','unit_price','unit_cost']],
        on='sku_id', how='left')
    phantom_events['est_revenue_at_risk'] = phantom_events['avg_on_hand'] * phantom_events['unit_price']
    phantom_by_sku = (
        phantom_events.groupby('sku_id')
        .agg(phantom_weeks=('is_phantom_candidate','sum'),
             avg_on_hand=('avg_on_hand','mean'),
             total_rev_risk=('est_revenue_at_risk','sum'))
        .sort_values('total_rev_risk', ascending=False)
        .reset_index()
    )
    phantom_by_sku = phantom_by_sku.merge(
        products[['sku_id','product_name','category','unit_price']], on='sku_id', how='left')
    phantom_sku_set    = set(phantom_by_sku['sku_id'].tolist())
    total_phantom_rev  = phantom_events['est_revenue_at_risk'].sum()
    return phantom_by_sku, phantom_sku_set, total_phantom_rev


def build_alert_df(latest_snap, demand_stats, phantom_sku_set,
                   active_promos, store_id,
                   CRITICAL_DAYS=3, WARNING_DAYS=7, MONITOR_DAYS=14,
                   TARGET_DAYS=14, TIER_WEIGHT=TIER_WEIGHT,
                   REV_RISK_DIV=1000, DOS_WEIGHT=5, SUPP_WEIGHT=20, SEASON_WEIGHT=10):
    """Objective 4: build alert dataframe with tiers + priority score."""
    alert_df = latest_snap.merge(
        demand_stats[['sku_id','product_name','category','is_perishable',
                      'avg_daily_demand','std_daily_demand','avg_demand_seasonal',
                      'safety_stock_seasonal','reorder_point_optimized',
                      'lead_time_final','avg_fill_rate','reliability_score',
                      'total_revenue','unit_price','unit_cost','pack_size',
                      'seasonal_factor','supplier_name']],
        on='sku_id', how='left')

    alert_df['days_of_supply_current'] = alert_df['days_of_supply'].fillna(
        alert_df['total_available'] / alert_df['avg_daily_demand'].replace(0, np.nan)).fillna(0)
    alert_df['demand_for_calc'] = alert_df['avg_demand_seasonal'].fillna(alert_df['avg_daily_demand'])

    def assign_alert(row):
        dos  = row['days_of_supply_current']
        hand = row['units_on_hand']
        tot  = row['total_available']
        rop  = row['reorder_point_optimized']
        if hand == 0 or dos <= CRITICAL_DAYS:
            return 'CRITICAL'
        elif dos <= WARNING_DAYS or tot <= rop:
            return 'WARNING'
        elif dos <= MONITOR_DAYS:
            return 'MONITOR'
        return 'OK'

    alert_df['alert_tier']      = alert_df.apply(assign_alert, axis=1)
    alert_df['units_needed']    = np.ceil(
        alert_df['demand_for_calc'] * (alert_df['lead_time_final'] + TARGET_DAYS)
        - alert_df['total_available']).clip(lower=0)
    alert_df['units_to_order']  = np.ceil(
        alert_df['units_needed'] / alert_df['pack_size'].replace(0,1)
    ) * alert_df['pack_size']
    alert_df['units_to_order']  = alert_df['units_to_order'].where(
        alert_df['alert_tier'].isin(['CRITICAL','WARNING']), other=0)
    alert_df['revenue_at_risk'] = (
        alert_df['demand_for_calc'] *
        (TARGET_DAYS - alert_df['days_of_supply_current'].clip(upper=TARGET_DAYS)) *
        alert_df['unit_price']).clip(lower=0)
    alert_df['revenue_at_risk'] = alert_df['revenue_at_risk'].where(
        alert_df['alert_tier'] != 'OK', other=0)
    alert_df['is_phantom'] = alert_df['sku_id'].isin(phantom_sku_set)

    # Promotion adjustment
    chain_promos = active_promos[active_promos['store_id'].isnull()][
        ['sku_id','demand_lift_factor']]
    store_promos = active_promos[active_promos['store_id'] == store_id][
        ['sku_id','demand_lift_factor']]
    all_promos = pd.concat([chain_promos, store_promos]).drop_duplicates('sku_id', keep='last')
    alert_df = alert_df.merge(all_promos, on='sku_id', how='left')
    alert_df['demand_lift_factor'] = alert_df['demand_lift_factor'].fillna(1.0)
    alert_df['is_on_promo']       = alert_df['demand_lift_factor'] > 1.0
    alert_df['units_to_order']    = np.where(
        alert_df['is_on_promo'],
        np.ceil(alert_df['units_to_order'] * alert_df['demand_lift_factor']),
        alert_df['units_to_order'])

    # 5-factor priority score
    alert_df['f1_tier']     = alert_df['alert_tier'].map(TIER_WEIGHT)
    alert_df['f2_revenue']  = alert_df['revenue_at_risk'] / REV_RISK_DIV
    max_dos = alert_df['days_of_supply_current'].clip(0, TARGET_DAYS).max()
    alert_df['f3_dos']      = (max_dos - alert_df['days_of_supply_current'].clip(0, TARGET_DAYS)) / max(max_dos, 1) * DOS_WEIGHT
    alert_df['f4_supplier'] = (1 - alert_df['reliability_score'].fillna(0.85).clip(0,1)) * SUPP_WEIGHT
    alert_df['f5_seasonal'] = (alert_df['seasonal_factor'].fillna(1.0) - 1) * SEASON_WEIGHT
    alert_df['priority_score'] = (
        alert_df['f1_tier'] + alert_df['f2_revenue'] + alert_df['f3_dos'] +
        alert_df['f4_supplier'] + alert_df['f5_seasonal'])

    alert_df['store_id'] = store_id
    alert_df = alert_df.sort_values('priority_score', ascending=False).reset_index(drop=True)
    return alert_df


def export_store(alert_df, demand_stats, phantom_by_sku, store_id, OUTPUT_DIR):
    """Export all CSVs for one store."""
    store_out = OUTPUT_DIR / store_id
    store_out.mkdir(exist_ok=True)
    export_cols = [
        'store_id','sku_id','product_name','category','is_perishable','is_phantom',
        'alert_tier','days_of_supply_current','units_on_hand','units_in_backroom',
        'total_available','reorder_point_optimized','safety_stock_seasonal',
        'avg_daily_demand','demand_for_calc','lead_time_final','reliability_score',
        'units_to_order','revenue_at_risk','priority_score',
        'f1_tier','f2_revenue','f3_dos','f4_supplier','f5_seasonal','snapshot_date'
    ]
    ok_cols = [c for c in export_cols if c in alert_df.columns]
    alert_df[ok_cols].to_csv(store_out / 'live_alert_report.csv', index=False)
    for tier in ['CRITICAL','WARNING','MONITOR']:
        alert_df[alert_df['alert_tier'] == tier][ok_cols].to_csv(
            store_out / f'alerts_{tier.lower()}.csv', index=False)
    rop_cols = ['sku_id','product_name','category','avg_daily_demand','std_daily_demand',
                'lead_time_final','safety_stock','safety_stock_optimized','safety_stock_seasonal',
                'ss_delta','ss_status','reorder_point','reorder_point_optimized','rop_delta']
    rop_ok = [c for c in rop_cols if c in demand_stats.columns]
    ds_out = demand_stats[rop_ok].copy()
    ds_out.insert(0, 'store_id', store_id)
    ds_out.to_csv(store_out / 'optimized_reorder_points.csv', index=False)
    if len(phantom_by_sku) > 0:
        ph_out = phantom_by_sku.copy()
        ph_out.insert(0, 'store_id', store_id)
        ph_out.to_csv(store_out / 'phantom_inventory.csv', index=False)


print('All functions defined. Ready to run the loop.')


All functions defined. Ready to run the loop.


---
## Section 3 — Main All-Store Loop
> Processes every store in sequence. Prints a one-line summary per store.
> All results are saved to `nexus_allstore_outputs/<store_id>/`
> and also accumulated into a network-level master dataframe.

In [4]:
from tqdm.auto import tqdm

network_alerts  = []   # one row per store-SKU
network_summary = []   # one row per store
failed_stores   = []

for store_id in tqdm(ALL_STORE_IDS, desc='Processing stores'):
    try:
        # ── Filter to this store ──────────────────────────────────
        s_sales  = sales[sales['store_id']         == store_id].copy()
        s_repl   = repl_logs[repl_logs['store_id'] == store_id].copy()
        s_snap   = inv_snap[inv_snap['store_id']   == store_id].copy()
        s_weekly = sales_weekly[sales_weekly['store_id'] == store_id].copy()
        store_info = stores[stores['store_id'] == store_id].iloc[0]

        if len(s_sales) == 0 or len(s_snap) == 0:
            failed_stores.append({'store_id': store_id, 'reason': 'no sales or snapshot data'})
            continue

        # ── Seasonal ──────────────────────────────────────────────
        monthly_idx, seasonal_factor = compute_seasonal(s_sales)

        # ── Demand stats ──────────────────────────────────────────
        demand_stats = compute_demand_stats(s_sales, prod_sup, seasonal_factor)
        if len(demand_stats) == 0:
            failed_stores.append({'store_id': store_id, 'reason': 'empty demand stats'})
            continue

        # ── Lead times ────────────────────────────────────────────
        demand_stats = compute_lead_times(s_repl, demand_stats)

        # ── Safety stock (Obj 3) ──────────────────────────────────
        demand_stats = compute_safety_stock(demand_stats, Z_SCORE, seasonal_factor)

        # ── Phantom detection ─────────────────────────────────────
        phantom_by_sku, phantom_sku_set, total_phantom_rev = compute_phantom(
            s_snap, s_weekly, products)

        # ── Latest inventory snapshot ─────────────────────────────
        latest_snap = (
            s_snap.sort_values('snapshot_date', ascending=False)
            .groupby('sku_id').first()
            .reset_index()
            [['sku_id','snapshot_date','units_on_hand','units_in_backroom',
              'days_of_supply','expiry_nearest_date']]
        )
        latest_snap['total_available'] = latest_snap['units_on_hand'] + latest_snap['units_in_backroom']

        # ── Alert engine (Obj 4) ──────────────────────────────────
        alert_df = build_alert_df(
            latest_snap, demand_stats, phantom_sku_set, active_promos, store_id)

        # ── Export per-store files ────────────────────────────────
        export_store(alert_df, demand_stats, phantom_by_sku, store_id, OUTPUT_DIR)

        # ── Accumulate ────────────────────────────────────────────
        network_alerts.append(alert_df)

        tier_counts = alert_df['alert_tier'].value_counts()
        network_summary.append({
            'store_id'       : store_id,
            'store_name'     : store_info['store_name'],
            'city'           : store_info['city'],
            'state'          : store_info['state'],
            'region'         : store_info['region'],
            'store_format'   : store_info['store_format'],
            'foot_traffic_tier': store_info['foot_traffic_tier'],
            'skus_analyzed'  : len(alert_df),
            'critical_count' : tier_counts.get('CRITICAL', 0),
            'warning_count'  : tier_counts.get('WARNING',  0),
            'monitor_count'  : tier_counts.get('MONITOR',  0),
            'ok_count'       : tier_counts.get('OK',       0),
            'revenue_at_risk': alert_df['revenue_at_risk'].sum(),
            'units_to_order' : alert_df['units_to_order'].sum(),
            'phantom_skus'   : len(phantom_by_sku),
            'phantom_rev_risk': total_phantom_rev,
            'understocked'   : (demand_stats['ss_status'] == 'UNDERSTOCKED').sum(),
            'overstocked'    : (demand_stats['ss_status'] == 'OVERSTOCKED').sum(),
            'optimal'        : (demand_stats['ss_status'] == 'OPTIMAL').sum(),
            'seasonal_factor': seasonal_factor,
        })

    except Exception as e:
        failed_stores.append({'store_id': store_id, 'reason': str(e)})
        continue

print(f'\nDone. {len(network_summary)} stores processed. {len(failed_stores)} failed.')
if failed_stores:
    print('Failed stores:')
    for f in failed_stores:
        print(f'  {f["store_id"]}: {f["reason"]}')


Processing stores:   0%|          | 0/478 [00:00<?, ?it/s]


Done. 478 stores processed. 0 failed.


---
## Section 4 — Network Master Files
> Combines all store results into two master CSVs:
> - `network_master_alerts.csv` — every store-SKU alert across all stores
> - `network_store_summary.csv` — one row per store, sortable by urgency

In [5]:
# ── Network-level master alert ────────────────────────────────────
if network_alerts:
    master_alerts = pd.concat(network_alerts, ignore_index=True)
    master_alerts.insert(0, 'report_time', NOW.strftime('%Y-%m-%d %H:%M:%S'))
    master_alerts.to_csv(OUTPUT_DIR / 'network_master_alerts.csv', index=False)
    print(f'network_master_alerts.csv  — {len(master_alerts):,} rows')

# ── Store summary ─────────────────────────────────────────────────
summary_df = pd.DataFrame(network_summary)
summary_df['urgency_score'] = (
    summary_df['critical_count'] * 1000 +
    summary_df['revenue_at_risk'] / 1000 +
    summary_df['warning_count'] * 100
)
summary_df = summary_df.sort_values('urgency_score', ascending=False).reset_index(drop=True)
summary_df.insert(0, 'report_time', NOW.strftime('%Y-%m-%d %H:%M:%S'))
summary_df.to_csv(OUTPUT_DIR / 'network_store_summary.csv', index=False)
print(f'network_store_summary.csv  — {len(summary_df)} stores')

# ── Network executive summary ─────────────────────────────────────
print()
print('='*62)
print('  NEXUS | NETWORK EXECUTIVE SUMMARY')
print('='*62)
print(f'  Stores processed     : {len(summary_df):,}')
print(f'  Total CRITICAL SKUs  : {summary_df["critical_count"].sum():,}')
print(f'  Total WARNING SKUs   : {summary_df["warning_count"].sum():,}')
print(f'  Total Revenue at Risk: ${summary_df["revenue_at_risk"].sum():,.0f}')
print(f'  Total Units to Order : {summary_df["units_to_order"].sum():,.0f}')
print(f'  Total Phantom SKUs   : {summary_df["phantom_skus"].sum():,}')
print()
print('TOP 10 MOST URGENT STORES:')
top10_cols = ['store_id','store_name','city','state','critical_count',
              'warning_count','revenue_at_risk','phantom_skus','urgency_score']
print(summary_df[top10_cols].head(10).to_string(index=False))


network_master_alerts.csv  — 38,240 rows
network_store_summary.csv  — 478 stores

  NEXUS | NETWORK EXECUTIVE SUMMARY
  Stores processed     : 478
  Total CRITICAL SKUs  : 6,544
  Total WARNING SKUs   : 6,463
  Total Revenue at Risk: $47,365,167
  Total Units to Order : 2,626,271
  Total Phantom SKUs   : 184

TOP 10 MOST URGENT STORES:
store_id              store_name          city state  critical_count  warning_count  revenue_at_risk  phantom_skus  urgency_score
   S0347     QuickPoint Honolulu      Honolulu    HI              33             15       249,208.89             0      34,749.21
   S0064    GoldenSquare Houston       Houston    TX              32             20       310,223.07             0      34,310.22
   S0169     RiverStation Aurora        Aurora    CO              30             18       240,011.83             0      32,040.01
   S0257 BrightHub San Francisco San Francisco    CA              30             13       152,179.74             0      31,452.18
   S0331    

---
## Section 5 — Network-Level Visualizations

In [6]:
if len(summary_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'Network Alert Overview — {NOW.strftime("%B %d, %Y %I:%M %p")} — {len(summary_df)} Stores',
                 fontsize=12, fontweight='bold')

    # Panel 1: Revenue at risk by region
    region_rar = summary_df.groupby('region')['revenue_at_risk'].sum().sort_values(ascending=True)
    axes[0].barh(region_rar.index, region_rar.values, color='#DC2626')
    axes[0].set_title('Revenue at Risk by Region')
    axes[0].set_xlabel('Revenue at Risk ($)')
    for i, v in enumerate(region_rar.values):
        axes[0].text(v+100, i, f'${v:,.0f}', va='center', fontsize=8)

    # Panel 2: CRITICAL counts by store format
    fmt_crit = summary_df.groupby('store_format')['critical_count'].sum().sort_values(ascending=True)
    axes[1].barh(fmt_crit.index, fmt_crit.values, color='#D97706')
    axes[1].set_title('CRITICAL Alerts by Store Format')
    axes[1].set_xlabel('CRITICAL SKU Count')

    # Panel 3: Top 15 stores by revenue at risk
    top15 = summary_df.nlargest(15, 'revenue_at_risk')
    labels = [f'{r["store_id"]} {r["city"]}' for _, r in top15.iterrows()]
    axes[2].barh(range(len(top15)), top15['revenue_at_risk'], color='#DC2626')
    axes[2].set_yticks(range(len(top15)))
    axes[2].set_yticklabels(labels, fontsize=7)
    axes[2].set_title('Top 15 Stores by Revenue at Risk')
    axes[2].set_xlabel('Revenue at Risk ($)')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'network_overview.png', bbox_inches='tight', dpi=130)
    plt.show()
    print('Saved: network_overview.png')


Saved: network_overview.png
